<a href="https://colab.research.google.com/github/Adarsha2004/finetuning/blob/main/gemma_cft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q unsloth
!pip install -q datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0

In [2]:
import torch
import re
import random
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True


In [3]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


[OK] Model loaded: unsloth/gemma-4-E2B-it
Parameters: 4,358,766,112


In [4]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """
    Calculates perplexity using a neutral assistant role.
    This avoids forcing the model into a 'summary' task and measures
    raw domain likelihood more accurately.
    """
    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts:
        # Use a neutral prompt to establish domain context without a task
        messages = [
            {"role": "user", "content": [{"type": "text", "text": "Provide medical textbook information:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]}
        ]

        full_input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_tensors="pt"
        ).to(model.device)

        prompt_input_ids = tokenizer.apply_chat_template(
            messages[:1],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        prompt_len = prompt_input_ids.shape[1]

        labels = full_input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad():
            outputs = model(input_ids=full_input_ids, labels=labels)

        num_tokens = (labels != -100).sum().item()
        if num_tokens > 0:
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    if total_tokens == 0: return float('inf')

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [5]:
# Re-evaluating general text with neutral prompt
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

ppl = compute_perplexity(model, tokenizer, general_texts)
print(f"General Perplexity (Neutral Prompt): {ppl:.2f}")

General Perplexity (Neutral Prompt): 232.13


In [6]:
sample_medical = [
    "The patient presented with acute chest pain radiating to the left arm, accompanied by diaphoresis and shortness of breath, suggestive of myocardial infarction.",
    "Hemoglobin levels were recorded at 8.2 g/dL, indicating moderate anemia, and iron supplementation was initiated.",
    "MRI findings revealed a lesion in the temporal lobe consistent with early-stage glioma.",
    "The individual reported persistent hyperglycemia with fasting blood glucose levels above 140 mg/dL, indicating uncontrolled diabetes mellitus.",
    "Antibiotic therapy with amoxicillin-clavulanate was prescribed for bacterial sinusitis following clinical evaluation.",
]

medical_ppl = compute_perplexity(model, tokenizer, sample_medical)

print(f"Medical Perplexity (Neutral Prompt): {medical_ppl:.1f}")
print("(This is our more accurate baseline for CPT)")

Medical Perplexity (Neutral Prompt): 139.7
(This is our more accurate baseline for CPT)


In [7]:
from datasets import load_dataset

configs = [
    "Pathology_Robbins",
    "Physiology_Levy",
    "Pharmacology_Katzung",
    "Neurology_Adams",
    "Pediatrics_Nelson"
]

NUM_TRAIN = 5000
NUM_VAL = 500
TARGET_TOTAL = NUM_TRAIN + NUM_VAL

all_texts = []

print(f"[...] Collecting {TARGET_TOTAL} samples...")

# -----------------------------
# LOAD DATA
# -----------------------------
for cfg in configs:
    print(f"\n[+] Loading: {cfg}")

    ds = load_dataset(
        "zxvix/MedicalTextbook",
        cfg,
        split="train",
        streaming=True
    )

    for example in ds:
        text = example.get("text", "")

        if text and len(text.split()) > 100:
            all_texts.append(text)

        if len(all_texts) >= TARGET_TOTAL:
            break

    if len(all_texts) >= TARGET_TOTAL:
        break

print(f"\n[OK] Total collected: {len(all_texts)}")

[...] Collecting 5500 samples...

[+] Loading: Pathology_Robbins


README.md: 0.00B [00:00, ?B/s]


[+] Loading: Physiology_Levy

[+] Loading: Pharmacology_Katzung

[+] Loading: Neurology_Adams

[OK] Total collected: 5500


In [8]:
sample = all_texts[0]
words = sample.split()
print(f"Sample filing length: {len(words)} words")
print(f"\nFirst 200 words:")
print(" ".join(words[:200]))
print("\n...")
print(f"\nLast 100 words:")
print(" ".join(words[-100:]))

Sample filing length: 498 words

First 200 words:
Plasma Membrane: Protection and Nutrient Acquisition Biosynthetic Machinery: Endoplasmic Reticulum and Golgi Apparatus Waste Disposal: Lysosomes and Proteasomes Modular Signaling Proteins, Hubs, and Components of the Extracellular Matrix Proliferation and the Cell Cycle Pathology literally translates to the study of suffering (Greek pathos = suffering, logos = study); as applied to modern medicine, it is the study of disease. Virchow was certainly correct in asserting that disease originates at the cellular level, but we now realize that cellular disturbances arise from alterations in molecules (genes, proteins, and others) that influence the survival and behavior of cells. Thus, the foundation of modern pathology is understanding the cellular and molecular abnormalities that give rise to diseases. It is helpful to consider these abnormalities in the context of normal cellular structure and function, which is the theme of this introduct

In [9]:
# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)

    # Fix merged words
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)

    # Fix commas spacing
    text = re.sub(r",([a-zA-Z])", r", \1", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_texts = [clean_text(t) for t in all_texts]


In [10]:
# -----------------------------
# SHUFFLE + SPLIT (IMPORTANT)
# -----------------------------
random.shuffle(cleaned_texts)

train_texts = cleaned_texts[:NUM_TRAIN]
val_texts   = cleaned_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"\nTrain filings: {len(train_texts)}")
print(f"Val filings:   {len(val_texts)}")


Train filings: 5000
Val filings:   500


In [11]:
# -----------------------------
# CHUNKING
# -----------------------------
def chunk_texts(texts, chunk_size=512, overlap=0.1):
    step = int(chunk_size * (1 - overlap))
    all_chunks = []

    for text in texts:
        words = text.split()
        for i in range(0, len(words), step):
            chunk = words[i:i + chunk_size]
            if len(chunk) > 50:
                all_chunks.append(" ".join(chunk))

    return all_chunks

train_chunks = chunk_texts(train_texts, chunk_size=512, overlap=0.1)
val_chunks   = chunk_texts(val_texts, chunk_size=512, overlap=0.1)

print(f"\nTrain chunks: {len(train_chunks)}")
print(f"Val chunks:   {len(val_chunks)}")


Train chunks: 7253
Val chunks:   730


In [12]:
# -----------------------------
# SAMPLE OUTPUT
# -----------------------------
sample = train_chunks[0].split()

print("\nSample chunk (first 100 words):")
print(" ".join(sample[:100]))

print("\n...")
print("\nLast 50 words:")
print(" ".join(sample[-50:]))


Sample chunk (first 100 words):
In the chronically demented patient, there are usually a number of “frontal release” signs, such as picking at the bedsheets and clothes, grasping, groping, sucking, and paratonic rigidity of the limbs. However, some demented patients are as bewildered as those with confusional psychosis, and the two conditions are distinguishable only by differences in their mode of onset and chronicity. This suggests that the affected parts of the nervous system may be the same in both conditions. At times, a left hemispheral lesion causing a mild Wernicke’s aphasia resembles a confusional state in that the stream of speech and thought are

...

Last 50 words:
for delirium, including those who have an underlying dementia, preexisting medical illnesses, or a history of alcoholism or serious depression. Furthermore, delirium is more common in males and, not surprisingly, is more likely when sensory function is already impaired (loss of vision and hearing) (Burns et al; 

In [13]:
# Convert to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 7253
})
Val dataset:   Dataset({
    features: ['text'],
    num_rows: 730
})


In [14]:
# IMPORTANT: Measure base model perplexity on the ACTUAL val chunks
# We need this before we reload the model for training
val_sample = val_chunks[:50]
base_ppl_val = compute_perplexity(model, tokenizer, val_sample)
print(f"Base model perplexity on medical validation chunks: {base_ppl_val:.1f}")
print("(We'll compare this to the CPT model later)")

Base model perplexity on medical validation chunks: 50.6
(We'll compare this to the CPT model later)


In [15]:
import gc
import time
import torch

# More robust memory recovery
try:
    del model
    del tokenizer
    del trainer
except NameError:
    pass

for _ in range(3):
    gc.collect()
    torch.cuda.empty_cache()

time.sleep(5) # Extended pause for VRAM stabilization

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    device_map="auto" # Explicitly let accelerate handle mapping
)

# --- ADD PEFT ADAPTERS ---
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "embed_tokens", "lm_head"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=True,
    loftq_config=None,
)

print(f"[OK] Model successfully reloaded and adapters attached.")

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


[OK] Model successfully reloaded and adapters attached.


In [16]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

training_args = UnslothTrainingArguments(
    output_dir="./cpt_medical",
    num_train_epochs=3,
    per_device_train_batch_size=4,     # Reduced from 16 to fit T4 VRAM
    gradient_accumulation_steps=8,      # Increased to keep effective batch size at 32
    learning_rate=2e-4,
    embedding_learning_rate=2e-5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    dataset_text_field="text",
    dataset_num_proc=2,
    eval_strategy="steps",
    eval_steps=50,                      # Less frequent eval saves time/memory
    per_device_eval_batch_size=4,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    seed=SEED,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("[OK] Trainer re-configured with memory-efficient settings.")

Unsloth: Sample packing skipped (processor-based model detected).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7253 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/730 [00:00<?, ? examples/s]

[OK] Trainer re-configured with memory-efficient settings.


In [ ]:
# TRAIN!
# On a T4 GPU with 135M model + 80 filings + 2 epochs, ~8-12 minutes
print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)
print("Watch the eval_loss -- that's your north star.")
print("It should decrease and then plateau.")
print()

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


TRAINING STARTED
Watch the eval_loss -- that's your north star.
It should decrease and then plateau.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,253 | Num Epochs = 3 | Total steps = 681
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 70,516,736 of 5,202,132,512 (1.36% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
50,3.921850,3.333673
100,3.070506,3.055936
150,2.922666,3.059342
200,2.822696,2.977335
250,2.626038,2.998034
300,2.570604,2.954993
350,2.550834,2.939641


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in ./cpt_medical/checkpoint-50/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")
Unsloth: Restored added_tokens_decoder metadata in ./cpt_medical/checkpoint-100/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `T